# IPN 13 lớp: V4 nhiều seed, so sánh có kiểm soát đầu symbolic (`kg_logits`)

Mục tiêu: trả lời "đầu symbolic có giúp dự đoán không" bằng so sánh **cặp** thay vì một lần chạy. Với mỗi seed, V4 được khởi tạo từ đúng checkpoint neural `c_ref_seed<seed>` (có vận tốc, masked_mean) rồi huấn luyện chung; ta so V4 `neural`, `kg_logits` và `final` với **chính checkpoint khởi tạo đó** trên validation, và so ensemble các V4 (final) với ensemble các checkpoint khởi tạo.

Vì dao động giữa các seed lớn (macro-F1 validation của C có độ lệch chuẩn khoảng 2,7 điểm), kết luận chỉ có nghĩa khi chênh lệch trung bình theo cặp lớn hơn dao động đó. Với mỗi seed, mọi quyết định dùng validation; `RUN_LOCKED_TEST=False` mặc định (test IPN đã bị mở nhiều lần, nếu bật là bằng chứng exploratory).

Điều kiện tiên quyết: đã chạy notebook sweep (có `outputs/ipn13_neural_sweep/c_ref_seed{42,123,3407}/best.pt` trên Drive) và code mới (có `--save-logits`, `ensemble_v4_logits.py`) đã được push.

In [ ]:
from pathlib import Path
import csv
import datetime as dt
import hashlib
import json
import shutil
import subprocess
import sys
import uuid
import zipfile

from google.colab import drive
drive.mount('/content/drive')

# Chỉ sửa các biến cấu hình ở đây. Không ghi raw keypoint/video vào artifact cuối.
DRIVE_DATA_ROOT = Path('/content/drive/MyDrive/ipn-extract-data')
# Artifact train/val c46: thư mục trên Drive (runs/ipn_symbolic_v2_post_c/train_val/c46e0e469d2cf3b6) hoặc file .zip tương đương.
TRAIN_VAL_ARTIFACT_ZIP = DRIVE_DATA_ROOT / 'runs' / 'ipn_symbolic_v2_post_c' / 'train_val' / 'c46e0e469d2cf3b6'
# Ablation C (masked_mean, light dropout); dùng run hoàn chỉnh, không dùng thư mục *_incomplete_* hay bigru_*.
NEURAL_SEED_ROOT = DRIVE_DATA_ROOT / 'outputs' / 'ipn13_neural_sweep'   # c_ref_seed<seed>/best.pt từ notebook sweep
SEEDS = [42, 123, 3407]
IPN_NEURAL_BASELINE_CHECKPOINT = NEURAL_SEED_ROOT / 'c_ref_seed42' / 'best.pt'   # chỉ để đọc metadata kiến trúc
V4_ROOT = DRIVE_DATA_ROOT / 'outputs' / 'ipn13_v4_multiseed'              # checkpoint V4 từng seed, giữ qua reconnect
IPN_V4_CHECKPOINT = None  # Path V4 13 lớp nếu chạy EVAL_ONLY.
EVAL_ONLY=False
RUN_LOCKED_TEST=False  # True = xem test lần thứ hai (exploratory): V4 neural/KG/final + neural-only, mỗi evaluator một lần.
IPN_NEURAL_CACHE_ROOT = None  # Tùy chọn; mặc định dò cache T<num_frames> trên Drive.

REPO_URL = 'https://github.com/HuyenHoang28/DynamicHandGestureWithXAI.git'
ARTIFACT_COMMIT = 'e817f16e0d504e95064b889288f06231bcf692ab'   # commit dựng graph của artifact c46
REPO_REF = 'HuyenHoang'   # nhánh chứa code mới (adapter V4 nhận vận tốc)
REPO_COMMIT = None   # None = đầu nhánh (cần code mới có --save-logits); hash đã dùng được ghi lại
EXPECTED_COUNTS = {'train': 2443, 'val': 589, 'test': 1101}
EXPECTED_NUM_CLASSES = 13
EXPECTED_TEMPLATE_IDS = set(range(EXPECTED_NUM_CLASSES))
RUN_ID = dt.datetime.now(dt.timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + uuid.uuid4().hex[:10]
WORK_ROOT = Path('/content') / f'ipn13_kg_logits_{RUN_ID}'
REPO_DIR = WORK_ROOT / 'repo'
ARTIFACT_EXTRACT_DIR = WORK_ROOT / 'train_val_artifact'
MANIFEST_DIR = WORK_ROOT / 'manifests'
V4_OUTPUT_DIR = WORK_ROOT / 'v4_train'
EVAL_OUTPUT_DIR = WORK_ROOT / 'evaluation'
LOCAL_RESULTS_DIR = WORK_ROOT / 'results'
DRIVE_RUN_ROOT = DRIVE_DATA_ROOT / 'runs' / 'ipn13_v4_multiseed'

assert EXPECTED_COUNTS == {'train': 2443, 'val': 589, 'test': 1101}
assert EXPECTED_NUM_CLASSES == 13
assert not (EVAL_ONLY and IPN_V4_CHECKPOINT is None), 'EVAL_ONLY cần IPN_V4_CHECKPOINT.'
print({'run_id': RUN_ID, 'eval_only': EVAL_ONLY, 'locked_test': RUN_LOCKED_TEST})


## 1. Clone revision cố định và chạy lệnh bằng argv

Mọi lệnh external đều là list argv qua subprocess.run(..., shell=False) mặc định. Revision phải resolve đúng e817f16; không chạy branch trôi nổi.

In [ ]:
COMMANDS = []
WORK_ROOT.mkdir(parents=True, exist_ok=True)

def run_argv(argv, cwd=None, capture_output=False):
    command = [str(value) for value in argv]
    COMMANDS.append(command)
    print('$', ' '.join(repr(value) for value in command))
    return subprocess.run(
        command,
        cwd=str(cwd) if cwd is not None else None,
        check=True,
        text=True,
        capture_output=capture_output,
    )

if not REPO_DIR.exists():
    run_argv(['git', 'clone', '--filter=blob:none', REPO_URL, REPO_DIR])
elif not (REPO_DIR / '.git').is_dir():
    raise RuntimeError(f'{REPO_DIR} tồn tại nhưng không phải Git repository.')
run_argv(['git', '-C', REPO_DIR, 'fetch', 'origin', REPO_COMMIT or REPO_REF])
run_argv(['git', '-C', REPO_DIR, 'checkout', '--detach', REPO_COMMIT or 'FETCH_HEAD'])
revision_result = run_argv(['git', '-C', REPO_DIR, 'rev-parse', 'HEAD'], capture_output=True)
REPO_COMMIT_RESOLVED = revision_result.stdout.strip()
if REPO_COMMIT:
    assert REPO_COMMIT_RESOLVED == REPO_COMMIT, (REPO_COMMIT_RESOLVED, REPO_COMMIT)

# Artifact c46 được dựng ở ARTIFACT_COMMIT. Đầu nhánh có thêm mã temporal factor trong predicates/events/kg/precompute,
# nhưng mặc định tắt: đã kiểm tra graph dựng ở đầu nhánh giống hệt từng byte với e817f16 (40 clip thật).
# Ở đây ta không bật --enable-temporal-factors nên graph test (nếu dựng) khớp với graph train/validation.
print('Artifact built at', ARTIFACT_COMMIT, '; code revision in use', REPO_COMMIT_RESOLVED)
adapter_source = (REPO_DIR / 'final' / 'code' / 'src_neurosymbolic' / 'models' / 'cross_attention_v2.py').read_text(encoding='utf-8')
assert 'use_motion_features' in adapter_source, f'Nhánh {REPO_REF} chưa có adapter V4 nhận vận tốc: hãy commit và push code mới trước.'

REQUIREMENTS_PATH = REPO_DIR / 'requirements.txt'
if REQUIREMENTS_PATH.is_file():
    run_argv([sys.executable, '-m', 'pip', 'install', '-q', '-r', REQUIREMENTS_PATH])

sys.path.insert(0, str(REPO_DIR / 'final' / 'code'))
sys.path.insert(0, str(REPO_DIR))
PRECOMPUTE_SCRIPT = REPO_DIR / 'final' / 'code' / 'src_neurosymbolic' / 'precompute_instance_graphs_phase_v2.py'
V4_TRAIN_SCRIPT = REPO_DIR / 'final' / 'code' / 'src_neurosymbolic' / 'train_cross_attention_v4.py'
V4_TEST_SCRIPT = REPO_DIR / 'final' / 'code' / 'src_neurosymbolic' / 'test_cross_attention_v2.py'
NEURAL_TEST_SCRIPT = REPO_DIR / 'final' / 'code' / 'neural_only_experiments' / 'test_neural_only.py'
for required_script in (PRECOMPUTE_SCRIPT, V4_TRAIN_SCRIPT, V4_TEST_SCRIPT, NEURAL_TEST_SCRIPT):
    assert required_script.is_file(), required_script
print('Repository revision:', REPO_COMMIT_RESOLVED)
for required in ('ensemble_v4_logits.py',):
    assert (REPO_DIR / 'scripts' / required).is_file(), f'{required} chưa có trên nhánh {REPO_REF}: hãy commit và push code mới.'
assert '--save-logits' in (REPO_DIR / 'final' / 'code' / 'src_neurosymbolic' / 'test_cross_attention_v2.py').read_text(encoding='utf-8'), 'test_cross_attention_v2.py chưa có --save-logits'


## 2. Giải nén và kiểm tra artifact c46

Artifact train/val là provenance đầu vào, không phải checkpoint V4. Các marker completed_run.json, run_config.json, code_revision.txt, số graph và 13 template đều là hard gate.

In [ ]:
def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def find_artifact_zip(configured):
    configured = Path(configured)
    if configured.exists():
        return configured.resolve()
    stem = configured.name.removesuffix('.zip')
    candidates = []
    if DRIVE_DATA_ROOT.is_dir():
        candidates = [path for path in DRIVE_DATA_ROOT.rglob(stem) if path.is_dir() and (path / 'completed_run.json').is_file()]
        candidates += sorted(DRIVE_DATA_ROOT.rglob(stem + '*.zip'))
    if len(candidates) != 1:
        raise FileNotFoundError(f'Không xác định được artifact train/val {stem}: {candidates}')
    return candidates[0].resolve()

def safe_extract(zip_path, destination):
    destination = Path(destination).resolve()
    if destination.exists() and any(destination.iterdir()):
        raise FileExistsError(f'Không overwrite thư mục giải nén: {destination}')
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as archive:
        names = [item.filename for item in archive.infolist()]
        for name in names:
            member = Path(name)
            if member.is_absolute() or '..' in member.parts:
                raise ValueError(f'ZIP member unsafe: {name}')
            target = (destination / name).resolve()
            try:
                target.relative_to(destination)
            except ValueError as exc:
                raise ValueError(f'ZIP member escapes destination: {name}') from exc
        archive.extractall(destination)
        roots = {Path(name).parts[0] for name in names if Path(name).parts}
    if len(roots) != 1:
        raise ValueError(f'Expected one artifact root, got {roots}')
    return destination / next(iter(roots))

TRAIN_VAL_ARTIFACT_ZIP = find_artifact_zip(TRAIN_VAL_ARTIFACT_ZIP)
if TRAIN_VAL_ARTIFACT_ZIP.is_dir():
    # Copy về local: graph test sẽ được ghi thêm vào đây và đọc Drive từng file rất chậm.
    ARTIFACT_EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
    ARTIFACT_ROOT = ARTIFACT_EXTRACT_DIR / TRAIN_VAL_ARTIFACT_ZIP.name
    if ARTIFACT_ROOT.exists():
        raise FileExistsError(f'Không overwrite thư mục artifact: {ARTIFACT_ROOT}')
    shutil.copytree(TRAIN_VAL_ARTIFACT_ZIP, ARTIFACT_ROOT)
else:
    assert TRAIN_VAL_ARTIFACT_ZIP.suffix.lower() == '.zip'
    ARTIFACT_ROOT = safe_extract(TRAIN_VAL_ARTIFACT_ZIP, ARTIFACT_EXTRACT_DIR)
COMPLETED_RUN_PATH = ARTIFACT_ROOT / 'completed_run.json'
RUN_CONFIG_PATH = ARTIFACT_ROOT / 'run_config.json'
CODE_REVISION_PATH = ARTIFACT_ROOT / 'code_revision.txt'
for marker in (COMPLETED_RUN_PATH, RUN_CONFIG_PATH, CODE_REVISION_PATH):
    assert marker.is_file(), marker
completed_run = json.loads(COMPLETED_RUN_PATH.read_text(encoding='utf-8'))
run_config = json.loads(RUN_CONFIG_PATH.read_text(encoding='utf-8'))
code_revision = CODE_REVISION_PATH.read_text(encoding='utf-8').strip()
assert completed_run.get('status') == 'complete'
assert completed_run.get('run_mode') == 'train_val'
assert completed_run.get('graph_counts') == {'train': 2443, 'val': 589}
assert code_revision == ARTIFACT_COMMIT, (code_revision, ARTIFACT_COMMIT)
assert run_config.get('commit') == ARTIFACT_COMMIT
assert run_config.get('active_splits') == ['train', 'val']
assert run_config.get('expected_counts') == EXPECTED_COUNTS
assert sha256_file(RUN_CONFIG_PATH) == completed_run['run_config_sha256']
payload_sha256 = completed_run.get('payload_sha256', {})
if payload_sha256:
    assert sha256_file(CODE_REVISION_PATH) == payload_sha256.get('code_revision.txt')
    template_yaml_path = ARTIFACT_ROOT / 'gesture_templates.yaml'
    if template_yaml_path.is_file() and payload_sha256.get('gesture_templates.yaml'):
        assert sha256_file(template_yaml_path) == payload_sha256['gesture_templates.yaml']

GRAPH_ROOT = ARTIFACT_ROOT / 'data' / 'instance_graphs'
TEMPLATE_ROOT = ARTIFACT_ROOT / 'data' / 'template_graphs'
def graph_files(split):
    return sorted((GRAPH_ROOT / split).rglob('*.json'))
assert len(graph_files('train')) == EXPECTED_COUNTS['train']
assert len(graph_files('val')) == EXPECTED_COUNTS['val']
assert len(graph_files('test')) == 0, 'Artifact train_val không được chứa test graph.'
ORIGINAL_MANIFESTS = ARTIFACT_ROOT / 'original_manifests'
assert (ORIGINAL_MANIFESTS / 'train.txt').is_file() and (ORIGINAL_MANIFESTS / 'val.txt').is_file()
assert len((ORIGINAL_MANIFESTS / 'train.txt').read_text(encoding='utf-8').splitlines()) == EXPECTED_COUNTS['train']
assert len((ORIGINAL_MANIFESTS / 'val.txt').read_text(encoding='utf-8').splitlines()) == EXPECTED_COUNTS['val']
template_jsons = sorted(path for path in TEMPLATE_ROOT.glob('*.json') if path.name != 'index.json')
template_index = json.loads((TEMPLATE_ROOT / 'index.json').read_text(encoding='utf-8'))
assert len(template_jsons) == EXPECTED_NUM_CLASSES
assert template_index.get('template_count') == EXPECTED_NUM_CLASSES
template_ids = {int(json.loads(path.read_text(encoding='utf-8'))['label_id']) for path in template_jsons}
assert template_ids == EXPECTED_TEMPLATE_IDS
assert not (ARTIFACT_ROOT / 'best.pt').exists(), 'Artifact train_val không được chứa V4 checkpoint.'
print({'artifact': str(TRAIN_VAL_ARTIFACT_ZIP), 'train_graphs': len(graph_files('train')), 'val_graphs': len(graph_files('val')), 'templates': len(template_jsons)})

## 3. Manifest và keypoint trên Drive

Manifest được tìm trong Drive, kiểm tra đủ 13 ID liên tục và được viết lại thành manifest local với đường dẫn keypoint đã resolve. Việc viết lại chỉ là metadata nhỏ; raw keypoint vẫn ở Drive.

In [ ]:
from src_neurosymbolic.models.keypoint_dataset import read_split_file

def manifest_candidates(name):
    preferred = [
        DRIVE_DATA_ROOT / name,
        DRIVE_DATA_ROOT / 'data' / name,
        DRIVE_DATA_ROOT / 'manifests' / name,
        DRIVE_DATA_ROOT / 'splits' / name,
    ]
    discovered = sorted(DRIVE_DATA_ROOT.rglob(name)) if DRIVE_DATA_ROOT.is_dir() else []
    return list(dict.fromkeys(path.resolve() for path in preferred + discovered if path.is_file()))

def source_label(source, fallback_split):
    parts = source.replace(chr(92), '/').split('/')
    split_index = parts.index(fallback_split) if fallback_split in parts else -1
    if split_index >= 0 and split_index + 1 < len(parts):
        return parts[split_index + 1]
    for split_name in ('train', 'val', 'test'):
        if split_name in parts and parts.index(split_name) + 1 < len(parts):
            return parts[parts.index(split_name) + 1]
    return None

def load_drive_manifest(split):
    valid = []
    for path in manifest_candidates(f'{split}.txt'):
        try:
            rows = read_split_file(path)
        except Exception:
            continue
        if len(rows) == EXPECTED_COUNTS[split] and {label for _, label in rows} == EXPECTED_TEMPLATE_IDS:
            valid.append((path, rows))
    if not valid:
        raise FileNotFoundError(f'Không tìm thấy manifest {split}.txt đúng {EXPECTED_COUNTS[split]} dòng/13 lớp.')
    preferred = [item for item in valid if item[0].parent in {DRIVE_DATA_ROOT.resolve(), (DRIVE_DATA_ROOT / 'data').resolve()}]
    if len(preferred) == 1:
        return preferred[0]
    if len(valid) != 1:
        raise RuntimeError(f'Multiple manifest candidates for {split}: {[str(path) for path, _ in valid]}')
    return valid[0]

DRIVE_MANIFESTS = {}
MANIFEST_ROWS = {}
for split in ('train', 'val', 'test'):
    DRIVE_MANIFESTS[split], MANIFEST_ROWS[split] = load_drive_manifest(split)
    ids = {label for _, label in MANIFEST_ROWS[split]}
    assert ids == EXPECTED_TEMPLATE_IDS, (split, ids)
    assert len(MANIFEST_ROWS[split]) == EXPECTED_COUNTS[split]

def resolve_keypoint_for_source(source, split, root):
    source_text = source.replace(chr(92), '/')
    parts = [part for part in source_text.split('/') if part]
    filename = parts[-1]
    candidates = []
    if 'keypoints' in parts:
        candidates.append(root.joinpath(*parts[parts.index('keypoints') + 1:]))
    if split in parts:
        split_index = parts.index(split)
        candidates.append(root.joinpath(*parts[split_index:]))
        if split_index + 2 < len(parts):
            candidates.append(root / split / parts[split_index + 1] / Path(*parts[split_index + 2:]))
    candidates.extend([root / split / filename, root / filename])
    for candidate in candidates:
        if candidate.is_file():
            return candidate.resolve()
    label = source_label(source, split)
    if label is not None:
        hits = list((root / split / label).rglob(filename)) if (root / split / label).is_dir() else []
        if len(hits) == 1:
            return hits[0].resolve()
    raise FileNotFoundError(f'Không resolve được keypoint: {source} dưới {root}')

keypoint_roots = [
    DRIVE_DATA_ROOT,
    DRIVE_DATA_ROOT / 'keypoints',
    DRIVE_DATA_ROOT / 'data' / 'keypoints',
]
KEYPOINT_ROOT = None
for candidate_root in keypoint_roots:
    if not candidate_root.is_dir():
        continue
    try:
        for split in ('train', 'val', 'test'):
            resolve_keypoint_for_source(MANIFEST_ROWS[split][0][0], split, candidate_root)
    except (FileNotFoundError, IndexError):
        continue
    KEYPOINT_ROOT = candidate_root.resolve()
    break
if KEYPOINT_ROOT is None:
    raise FileNotFoundError('Không tìm thấy keypoint root chứa cả train/val/test trên Drive.')

def materialize_manifest(split):
    MANIFEST_DIR.mkdir(parents=True, exist_ok=True)
    output = MANIFEST_DIR / f'{split}.txt'
    with output.open('w', encoding='utf-8') as handle:
        for source, label in MANIFEST_ROWS[split]:
            resolved = resolve_keypoint_for_source(source, split, KEYPOINT_ROOT)
            handle.write(f'{resolved} 0 {label}\n')
    return output

TRAIN_FILE = materialize_manifest('train')
VAL_FILE = materialize_manifest('val')
TEST_FILE = materialize_manifest('test')
MANIFEST_SHA256 = {split: sha256_file(path) for split, path in [('train', TRAIN_FILE), ('val', VAL_FILE), ('test', TEST_FILE)]}
print({'keypoint_root': str(KEYPOINT_ROOT), 'manifests': {key: str(value) for key, value in DRIVE_MANIFESTS.items()}, 'normalized_hashes': MANIFEST_SHA256})

## 4. Đọc metadata baseline trước khi dựng V4

V4 phải có cùng d_model, layers, heads, dropout, num_frames, vận tốc và pooling với checkpoint baseline. Không hardcode cấu hình cũ 128/3/4. Adapter neural của V4 nhận vận tốc và cả ba kiểu pooling (`mean`, `masked_mean`, `attention`), nên checkpoint C (192/4/8, vận tốc, masked_mean) nạp được nguyên vẹn; neural-only vẫn được đánh giá riêng bằng evaluator của nó.

In [ ]:
import torch
from src_neurosymbolic.models.cross_attention_v2 import load_checkpoint_compat

def checkpoint_path(value, name):
    if value is None:
        raise ValueError(f'{name} chưa được cấu hình.')
    path = Path(value).expanduser()
    if not path.is_file():
        raise FileNotFoundError(f'{name} không tồn tại: {path}')
    return path.resolve()

if IPN_NEURAL_BASELINE_CHECKPOINT is None:
    # Tự dò checkpoint neural-only 13 lớp tương thích V4 trên Drive; mơ hồ thì dừng và liệt kê.
    found = []
    for candidate in sorted(DRIVE_DATA_ROOT.rglob('*.pt')):
        if 'v4' in str(candidate).lower() or candidate.parent.name == 'payload':
            continue
        try:
            probe = load_checkpoint_compat(candidate)
            a = probe.get('args', {}) or {}
        except Exception:
            continue
        motion = bool(a['use_motion_features']) if 'use_motion_features' in a else not bool(a.get('disable_motion_features', False))
        if (a.get('backbone', 'transformer') == 'transformer' and int(a.get('num_classes', -1)) == EXPECTED_NUM_CLASSES
                and not motion and str(a.get('pooling', 'mean')) in {'mean', 'masked_mean'}):
            found.append((candidate, a.get('d_model'), a.get('layers'), a.get('heads'), a.get('pooling', 'mean'), probe.get('epoch')))
    for row in found:
        print(row)
    if len(found) != 1:
        raise RuntimeError(f'Có {len(found)} checkpoint tương thích; gán IPN_NEURAL_BASELINE_CHECKPOINT thủ công ở cell cấu hình (khuyến nghị ablation C: d_model=192, 4 layer, 8 head, masked_mean).')
    IPN_NEURAL_BASELINE_CHECKPOINT = found[0][0]

IPN_NEURAL_BASELINE_CHECKPOINT = checkpoint_path(IPN_NEURAL_BASELINE_CHECKPOINT, 'IPN_NEURAL_BASELINE_CHECKPOINT')
baseline_payload = load_checkpoint_compat(IPN_NEURAL_BASELINE_CHECKPOINT)
BASELINE_ARGS = dict(baseline_payload.get('args', {}))
if not BASELINE_ARGS:
    raise ValueError('Baseline checkpoint thiếu args; không thể chứng minh kiến trúc tương thích.')
BASELINE_BACKBONE = BASELINE_ARGS.get('backbone', 'transformer')
BASELINE_NUM_CLASSES = int(BASELINE_ARGS.get('num_classes', -1))
BASELINE_NUM_FRAMES = int(BASELINE_ARGS.get('num_frames', 64))
BASELINE_D_MODEL = int(BASELINE_ARGS.get('d_model', -1))
BASELINE_LAYERS = int(BASELINE_ARGS.get('layers', -1))
BASELINE_HEADS = int(BASELINE_ARGS.get('heads', -1))
BASELINE_DROPOUT = float(BASELINE_ARGS.get('dropout', 0.1))
BASELINE_POOLING = str(BASELINE_ARGS.get('pooling', 'mean'))
if 'use_motion_features' in BASELINE_ARGS:
    BASELINE_MOTION_FEATURES = bool(BASELINE_ARGS['use_motion_features'])
else:
    BASELINE_MOTION_FEATURES = not bool(BASELINE_ARGS.get('disable_motion_features', False))

assert BASELINE_BACKBONE == 'transformer', f'V4 neural adapter chỉ hỗ trợ transformer, nhận {BASELINE_BACKBONE!r}'
assert BASELINE_NUM_CLASSES == EXPECTED_NUM_CLASSES, f'Baseline phải có 13 logits, nhận {BASELINE_NUM_CLASSES}'
assert BASELINE_NUM_FRAMES > 0 and BASELINE_D_MODEL > 0 and BASELINE_LAYERS > 0 and BASELINE_HEADS > 0
assert BASELINE_D_MODEL % BASELINE_HEADS == 0, 'd_model phải chia hết cho heads.'
print('Neural baseline (C) dùng velocity:', BASELINE_MOTION_FEATURES, '- V4 nhận cùng đặc trưng này qua --neural-motion-features (bước 4b).')
assert BASELINE_POOLING in {'mean', 'masked_mean', 'attention'}, f'Pooling không hỗ trợ: {BASELINE_POOLING}'
baseline_state = {key.removeprefix('module.'): value for key, value in baseline_payload.get('model', baseline_payload).items()}
input_weight = baseline_state.get('input_proj.0.weight')
classifier_weight = baseline_state.get('classifier.4.weight')
assert input_weight is not None and tuple(input_weight.shape)[1] == 52 * (5 if BASELINE_MOTION_FEATURES else 3), 'input_proj không khớp metadata motion.'
assert classifier_weight is not None and tuple(classifier_weight.shape)[0] == EXPECTED_NUM_CLASSES

BASELINE_CACHE_ROOT = Path(IPN_NEURAL_CACHE_ROOT).expanduser() if IPN_NEURAL_CACHE_ROOT else None
if BASELINE_CACHE_ROOT is None:
    saved_cache = BASELINE_ARGS.get('cache_root')
    saved_cache_path = Path(saved_cache).expanduser() if saved_cache else None
    candidates = [saved_cache_path, DRIVE_DATA_ROOT / 'keypoint_tensor_cache', DRIVE_DATA_ROOT / 'data' / 'keypoint_tensor_cache']
    BASELINE_CACHE_ROOT = next((path.resolve() for path in candidates if path is not None and path.is_dir()), None)
if BASELINE_CACHE_ROOT is None:
    raise FileNotFoundError('Neural-only evaluator cần keypoint tensor cache; gán IPN_NEURAL_CACHE_ROOT trên Drive.')
print({'baseline': str(IPN_NEURAL_BASELINE_CHECKPOINT), 'd_model': BASELINE_D_MODEL, 'layers': BASELINE_LAYERS, 'heads': BASELINE_HEADS, 'dropout': BASELINE_DROPOUT, 'num_frames': BASELINE_NUM_FRAMES, 'pooling': BASELINE_POOLING, 'cache_root': str(BASELINE_CACHE_ROOT)})

## 4b. Checkpoint khởi tạo của từng seed

Mọi seed phải cùng kiến trúc với checkpoint tham chiếu (d_model, số lớp, head, pooling, vận tốc, số lớp đầu ra); nếu lệch, dừng ngay thay vì nạp sai.

In [ ]:
SEED_INITS = {seed: checkpoint_path(NEURAL_SEED_ROOT / f'c_ref_seed{seed}' / 'best.pt', f'neural c_ref seed {seed}') for seed in SEEDS}
for seed, path in SEED_INITS.items():
    seed_args = load_checkpoint_compat(path).get('args', {})
    for key in ('d_model', 'layers', 'heads', 'pooling', 'num_classes', 'num_frames'):
        assert seed_args.get(key) == BASELINE_ARGS.get(key), (seed, key, seed_args.get(key), BASELINE_ARGS.get(key))
    assert bool(seed_args.get('disable_motion_features', False)) == bool(BASELINE_ARGS.get('disable_motion_features', False))
    assert bool(seed_args.get('hand_features', False)) == bool(BASELINE_ARGS.get('hand_features', False))
V4_NEURAL_ARGS = []
if BASELINE_MOTION_FEATURES:
    V4_NEURAL_ARGS.append('--neural-motion-features')
V4_NEURAL_ARGS += ['--neural-pooling', BASELINE_POOLING]
if BASELINE_ARGS.get('hand_features'):
    V4_NEURAL_ARGS.append('--neural-hand-features')
print({'seeds': SEEDS, 'neural_args': V4_NEURAL_ARGS, 'inits': {s: str(p) for s, p in SEED_INITS.items()}})


## 5. Test graph (chỉ khi bật test)

Chỉ dựng 1.101 graph test khi `RUN_LOCKED_TEST=True`; ở chế độ mặc định chỉ cần graph train/validation có sẵn trong artifact.

In [ ]:
if RUN_LOCKED_TEST:
    BUILD_SPLITS = ['test']
    assert set(BUILD_SPLITS) == {'test'}
    assert len(graph_files('train')) == EXPECTED_COUNTS['train']
    assert len(graph_files('val')) == EXPECTED_COUNTS['val']
    test_graphs_before = len(graph_files('test'))
    assert test_graphs_before <= EXPECTED_COUNTS['test']
    event_config = dict(run_config['event_config'])
    predicate_thresholds = dict(run_config['predicate_thresholds'])
    required_event_keys = ('max_phase_duration', 'min_phase_duration', 'min_overlap_ratio', 'min_overlap_frames')
    assert all(key in event_config for key in required_event_keys)
    required_predicate_keys = ('hand', 'finger', 'pinch', 'trend_consistency')
    assert all(key in predicate_thresholds for key in required_predicate_keys)

    missing_test_graphs = EXPECTED_COUNTS['test'] - test_graphs_before
    if missing_test_graphs:
        precompute_command = [
            sys.executable, '-u', PRECOMPUTE_SCRIPT,
            '--keypoint-root', KEYPOINT_ROOT,
            '--output-root', GRAPH_ROOT,
            '--num-frames', BASELINE_NUM_FRAMES,
            '--train-file', TRAIN_FILE,
            '--val-file', VAL_FILE,
            '--test-file', TEST_FILE,
            '--splits', 'test',
            '--max-phase-duration', event_config['max_phase_duration'],
            '--min-phase-duration', event_config['min_phase_duration'],
            '--min-overlap-ratio', event_config['min_overlap_ratio'],
            '--min-overlap-frames', event_config['min_overlap_frames'],
            '--hand-motion-threshold', predicate_thresholds['hand'],
            '--finger-motion-threshold', predicate_thresholds['finger'],
            '--pinch-threshold', predicate_thresholds['pinch'],
            '--trend-consistency', predicate_thresholds['trend_consistency'],
        ]
        run_argv(precompute_command, cwd=REPO_DIR)
    else:
        print('Test graph đã đủ; không chạy lại precompute.')

    assert len(graph_files('train')) == EXPECTED_COUNTS['train']
    assert len(graph_files('val')) == EXPECTED_COUNTS['val']
    assert len(graph_files('test')) == EXPECTED_COUNTS['test']
    print({'test_graphs_before': test_graphs_before, 'test_graphs_missing': missing_test_graphs, 'test_graphs_after': len(graph_files('test')), 'event_config': event_config, 'predicate_thresholds': predicate_thresholds})
else:
    print('RUN_LOCKED_TEST=False: không dựng test graph.')
    GRAPH_ROOT = ARTIFACT_ROOT / 'data' / 'instance_graphs'


In [ ]:
import statistics
NEURAL_DIR_FOR_ENSEMBLE = REPO_DIR / 'final' / 'code' / 'neural_only_experiments'


## 6. Huấn luyện V4 từng seed

Mỗi seed khởi tạo từ `c_ref_seed<seed>`, đóng băng encoder 3 epoch đầu để nhánh KG khởi động, augmentation giống neural C (không frame masking), patience 12 vì validation dao động. Không dùng test; `best.pt` theo macro-F1 validation. Run đã xong được dùng lại; run dở dang đổi tên, không xóa.

In [ ]:
DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'


def run_logged(argv, cwd=None, log_path=None):
    command = [str(v) for v in argv]
    COMMANDS.append(command)
    print('$', ' '.join(command))
    log = open(log_path, 'a', encoding='utf-8') if log_path else None
    try:
        process = subprocess.Popen(command, cwd=str(cwd) if cwd else None, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line, end='')
            if log:
                log.write(line)
        if process.wait() != 0:
            raise subprocess.CalledProcessError(process.returncode, command)
    finally:
        if log:
            log.close()


V4_ROOT.mkdir(parents=True, exist_ok=True)
V4_RUNS = {}
for seed in SEEDS:
    run_dir = V4_ROOT / f'seed{seed}'
    V4_RUNS[seed] = run_dir
    marker = run_dir / 'V4_RUN_COMPLETE.json'
    if marker.is_file() and (run_dir / 'best.pt').is_file():
        print('Dùng lại run V4 đã hoàn tất:', run_dir.name)
        continue
    if run_dir.exists() and any(run_dir.iterdir()):
        stale = run_dir.with_name(run_dir.name + '_incomplete_' + dt.datetime.now().strftime('%Y%m%dT%H%M%S'))
        run_dir.rename(stale)
        print('Run dở dang được đổi tên thành', stale.name)
    command = [
        sys.executable, '-u', V4_TRAIN_SCRIPT,
        '--keypoint-root', KEYPOINT_ROOT, '--instance-graph-root', GRAPH_ROOT, '--template-dir', TEMPLATE_ROOT,
        '--neural-checkpoint', SEED_INITS[seed], *V4_NEURAL_ARGS,
        '--train-file', TRAIN_FILE, '--val-file', VAL_FILE, '--output-dir', run_dir,
        '--logit-branch', 'final', '--num-classes', EXPECTED_NUM_CLASSES, '--num-frames', BASELINE_NUM_FRAMES,
        '--d-model', BASELINE_D_MODEL, '--layers', BASELINE_LAYERS, '--heads', BASELINE_HEADS, '--dropout', BASELINE_DROPOUT,
        '--batch-size', 32, '--epochs', 50, '--select-metric', 'macro_f1',
        '--freeze-neural-epochs', 3, '--early-stop-patience', 12, '--scheduler-patience', 4,
        '--frame-mask-probability', 0, '--keypoint-dropout-probability', 0.02,
        '--device', DEVICE, '--seed', seed,
    ]
    assert '--test-file' not in command and '--save-best-test' not in command
    run_dir.mkdir(parents=True)
    run_logged(command, cwd=REPO_DIR, log_path=run_dir / 'train.log')
    assert (run_dir / 'best.pt').is_file(), 'V4 không tạo best.pt theo validation.'
    marker.write_text(json.dumps({'seed': seed, 'commit': REPO_COMMIT_RESOLVED, 'init': str(SEED_INITS[seed]),
                                  'command': [str(v) for v in command]}, indent=2), encoding='utf-8')
print('Xong huấn luyện V4:', len(V4_RUNS), 'seed')


## 7. Đánh giá validation: V4 so với checkpoint khởi tạo, cùng seed

V4 xuất đồng thời ba đầu ra (`neural`, `kg`, `final`) và lưu logits để ensemble. Checkpoint khởi tạo được đánh giá bằng evaluator neural-only độc lập, trên cùng 589 clip.

In [ ]:
def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))


def cache_args():
    return ['--cache-root', BASELINE_CACHE_ROOT] if BASELINE_CACHE_ROOT else []


VAL_EVAL = {}
for seed in SEEDS:
    out_dir = V4_RUNS[seed] / 'val_eval'
    v4_metrics = out_dir / 'v4_metrics.json'
    init_metrics = out_dir / 'init_metrics.json'
    if not v4_metrics.is_file():
        run_logged([sys.executable, '-u', V4_TEST_SCRIPT, '--checkpoint', V4_RUNS[seed] / 'best.pt',
                    '--keypoint-root', KEYPOINT_ROOT, '--instance-graph-root', GRAPH_ROOT, '--template-dir', TEMPLATE_ROOT,
                    '--test-file', VAL_FILE, '--split', 'val', '--batch-size', 32, '--num-workers', 0, '--device', DEVICE,
                    '--timing-warmup-batches', 1, '--timing-repeats', 1, '--eval-branch', 'final', '--save-logits',
                    '--output', v4_metrics, *cache_args()], cwd=REPO_DIR)
    if not init_metrics.is_file():
        run_logged([sys.executable, '-u', NEURAL_TEST_SCRIPT, '--checkpoint', SEED_INITS[seed], '--cache-root', BASELINE_CACHE_ROOT,
                    '--test-file', DRIVE_MANIFESTS['val'], '--split', 'val', '--batch-size', 128, '--num-workers', 0,
                    '--device', DEVICE, '--timing-warmup-batches', 1, '--timing-repeats', 1, '--output', init_metrics],
                   cwd=REPO_DIR / 'final' / 'code' / 'neural_only_experiments')
    VAL_EVAL[seed] = {'v4': read_json(v4_metrics), 'init': read_json(init_metrics), 'dir': out_dir}
print('Đánh giá validation xong.')


## 8. Bảng so sánh cặp và kết luận

Cột Δ là V4 trừ checkpoint khởi tạo của **cùng seed**. "KG sửa/hại" đếm clip mà `kg_logits` đúng/sai khi `neural` của V4 sai/đúng. Đầu symbolic chỉ được coi là có ích nếu Δ trung bình dương và lớn hơn độ lệch chuẩn giữa các seed.

In [ ]:
rows = []
for seed in SEEDS:
    v4 = VAL_EVAL[seed]['v4']
    init = VAL_EVAL[seed]['init']
    branches = v4['branches']
    counts = v4['neural_vs_kg']['counts']
    rows.append({
        'seed': seed,
        'init_acc': init['accuracy'], 'init_f1': init['macro_f1'],
        'neural_acc': branches['neural']['accuracy'], 'neural_f1': branches['neural']['macro_f1'],
        'kg_acc': branches['kg']['accuracy'], 'kg_f1': branches['kg']['macro_f1'],
        'final_acc': branches['final']['accuracy'], 'final_f1': branches['final']['macro_f1'],
        'kg_fix': counts.get('kg_fix', 0), 'kg_harm': counts.get('kg_harm', 0),
        'gate': v4.get('gate_mean'),
    })
print(f"{'seed':>5} {'init acc/F1':>14} {'V4 neural':>14} {'V4 kg':>14} {'V4 final':>14} {'Δfinal acc/F1':>16} {'kg +/-':>8} {'gate':>6}")
for r in rows:
    print(f"{r['seed']:>5} {r['init_acc']:.4f}/{r['init_f1']:.4f} {r['neural_acc']:.4f}/{r['neural_f1']:.4f} {r['kg_acc']:.4f}/{r['kg_f1']:.4f} "
          f"{r['final_acc']:.4f}/{r['final_f1']:.4f} {r['final_acc'] - r['init_acc']:+.4f}/{r['final_f1'] - r['init_f1']:+.4f} "
          f"{r['kg_fix']:>3}/{r['kg_harm']:<3} {r['gate']:.2f}")
delta_f1 = [r['final_f1'] - r['init_f1'] for r in rows]
delta_acc = [r['final_acc'] - r['init_acc'] for r in rows]
init_sd = statistics.stdev([r['init_f1'] for r in rows]) if len(rows) > 1 else 0.0
print(f"\nΔ(final - init): macro-F1 {statistics.mean(delta_f1):+.4f} ± {statistics.stdev(delta_f1) if len(rows) > 1 else 0.0:.4f}, "
      f"accuracy {statistics.mean(delta_acc):+.4f}")
print(f"Dao động macro-F1 giữa các checkpoint khởi tạo (sd): {init_sd:.4f}")
net_kg = sum(r['kg_fix'] - r['kg_harm'] for r in rows)
print(f"KG net (sửa - hại) cộng qua các seed: {net_kg:+d}")
if len(rows) >= 3 and all(d > 0 for d in delta_f1) and statistics.mean(delta_f1) > 0.005:
    print('=> Dấu hiệu đầu symbolic giúp: mọi seed đều tăng macro-F1 so với checkpoint khởi tạo. Cần xác nhận bằng ensemble và test.')
else:
    print('=> Chưa có bằng chứng đầu symbolic giúp: không phải seed nào cũng tăng, hoặc mức tăng trung bình quá nhỏ so với dao động giữa các seed.')

CLICK_PINCH = ['G01', 'G02', 'G08', 'G09', 'G10', 'G11']
print('\nF1 từng lớp (trung bình các seed): checkpoint khởi tạo / V4 final / V4 kg')
for name in CLICK_PINCH + ['B0A', 'B0B']:
    per = {'init': [], 'final': [], 'kg': []}
    for seed in SEEDS:
        for row in VAL_EVAL[seed]['init']['per_class']:
            if row['label_name'] == name:
                per['init'].append(row['f1'])
        with (VAL_EVAL[seed]['dir'] / 'test_per_class.csv').open(encoding='utf-8', newline='') as handle:
            for row in csv.DictReader(handle):
                if row['class_name'] == name:
                    per['final'].append(float(row['final_f1']))
                    per['kg'].append(float(row['kg_f1']))
    if not per['init'] or not per['final']:
        continue
    print(f"  {name}: {statistics.mean(per['init']):.3f} / {statistics.mean(per['final']):.3f} / {statistics.mean(per['kg']):.3f}")


## 9. Ensemble: V4 (có đầu symbolic) so với neural

Ensemble các V4 (trung bình xác suất của `final`) so với ensemble các checkpoint khởi tạo, cùng validation. Nếu V4 ensemble không hơn neural ensemble thì đầu symbolic không thêm giá trị ở mức ensemble.

In [ ]:
ENS_DIR = V4_ROOT / 'ensembles'
logits_files = [VAL_EVAL[seed]['dir'] / 'test_logits.pt' for seed in SEEDS]
v4_ens = {}
for branch in ('final', 'kg', 'neural'):
    out = ENS_DIR / f'v4_{branch}_val.json'
    if not out.is_file():
        run_logged([sys.executable, REPO_DIR / 'scripts' / 'ensemble_v4_logits.py', '--logits', *logits_files,
                    '--branch', branch, '--output', out], cwd=REPO_DIR)
    v4_ens[branch] = read_json(out)
init_out = ENS_DIR / 'init_neural_val' / 'metrics.json'
if not init_out.is_file():
    run_logged([sys.executable, '-u', NEURAL_DIR_FOR_ENSEMBLE / 'ensemble_neural_only.py', '--checkpoints', *[SEED_INITS[s] for s in SEEDS],
                '--cache-root', BASELINE_CACHE_ROOT, '--test-file', DRIVE_MANIFESTS['val'], '--split', 'val', '--device', DEVICE,
                '--output', init_out], cwd=NEURAL_DIR_FOR_ENSEMBLE)
init_ens = read_json(init_out)
print(f"{'model':28} {'acc':>8} {'macro-F1':>9}")
print(f"{'neural-only init (ensemble)':28} {init_ens['accuracy']:8.4f} {init_ens['macro_f1']:9.4f}")
for branch in ('neural', 'kg', 'final'):
    print(f"{'V4 ' + branch + ' (ensemble)':28} {v4_ens[branch]['accuracy']:8.4f} {v4_ens[branch]['macro_f1']:9.4f}")
print(f"\nV4 final ensemble - neural ensemble: accuracy {v4_ens['final']['accuracy'] - init_ens['accuracy']:+.4f}, "
      f"macro-F1 {v4_ens['final']['macro_f1'] - init_ens['macro_f1']:+.4f}")


## 10. Test (khóa mặc định)

Chỉ khi `RUN_LOCKED_TEST=True`: đánh giá từng V4, checkpoint khởi tạo và hai ensemble trên 1.101 clip test. Test IPN đã bị mở nhiều lần, nên đây chỉ là bằng chứng exploratory. Không đổi cấu hình rồi chạy lại.

In [ ]:
TEST_RESULTS = {}
if RUN_LOCKED_TEST:
    for seed in SEEDS:
        out_dir = V4_RUNS[seed] / 'test_eval'
        assert not (out_dir / 'v4_metrics.json').exists(), f'Đã có kết quả test cho seed {seed}; không chạy lại.'
        run_logged([sys.executable, '-u', V4_TEST_SCRIPT, '--checkpoint', V4_RUNS[seed] / 'best.pt',
                    '--keypoint-root', KEYPOINT_ROOT, '--instance-graph-root', GRAPH_ROOT, '--template-dir', TEMPLATE_ROOT,
                    '--test-file', TEST_FILE, '--split', 'test', '--batch-size', 32, '--num-workers', 0, '--device', DEVICE,
                    '--timing-warmup-batches', 1, '--timing-repeats', 1, '--eval-branch', 'final', '--save-logits',
                    '--output', out_dir / 'v4_metrics.json', *cache_args()], cwd=REPO_DIR)
        run_logged([sys.executable, '-u', NEURAL_TEST_SCRIPT, '--checkpoint', SEED_INITS[seed], '--cache-root', BASELINE_CACHE_ROOT,
                    '--test-file', DRIVE_MANIFESTS['test'], '--split', 'test', '--batch-size', 128, '--num-workers', 0,
                    '--device', DEVICE, '--timing-warmup-batches', 1, '--timing-repeats', 1, '--output', out_dir / 'init_metrics.json'],
                   cwd=NEURAL_DIR_FOR_ENSEMBLE)
        TEST_RESULTS[seed] = {'v4': read_json(out_dir / 'v4_metrics.json'), 'init': read_json(out_dir / 'init_metrics.json'), 'dir': out_dir}
    test_logits = [TEST_RESULTS[seed]['dir'] / 'test_logits.pt' for seed in SEEDS]
    for branch in ('final', 'kg', 'neural'):
        run_logged([sys.executable, REPO_DIR / 'scripts' / 'ensemble_v4_logits.py', '--logits', *test_logits, '--branch', branch,
                    '--output', ENS_DIR / f'v4_{branch}_test.json'], cwd=REPO_DIR)
    run_logged([sys.executable, '-u', NEURAL_DIR_FOR_ENSEMBLE / 'ensemble_neural_only.py', '--checkpoints', *[SEED_INITS[s] for s in SEEDS],
                '--cache-root', BASELINE_CACHE_ROOT, '--test-file', DRIVE_MANIFESTS['test'], '--split', 'test', '--device', DEVICE,
                '--output', ENS_DIR / 'init_neural_test' / 'metrics.json'], cwd=NEURAL_DIR_FOR_ENSEMBLE)
    print(f"{'seed':>5} {'init acc/F1':>16} {'V4 final acc/F1':>18} {'V4 kg acc/F1':>16}")
    for seed in SEEDS:
        v4 = TEST_RESULTS[seed]['v4']['branches']
        init = TEST_RESULTS[seed]['init']
        print(f"{seed:>5} {init['accuracy']:.4f}/{init['macro_f1']:.4f}   {v4['final']['accuracy']:.4f}/{v4['final']['macro_f1']:.4f}   {v4['kg']['accuracy']:.4f}/{v4['kg']['macro_f1']:.4f}")
    t_init = read_json(ENS_DIR / 'init_neural_test' / 'metrics.json')
    t_final = read_json(ENS_DIR / 'v4_final_test.json')
    print(f"ensemble: neural init {t_init['accuracy']:.4f}/{t_init['macro_f1']:.4f}  V4 final {t_final['accuracy']:.4f}/{t_final['macro_f1']:.4f}")
else:
    print('RUN_LOCKED_TEST=False: test chưa được chạy.')


## 11. Đóng gói kết quả vào một ZIP duy nhất

Gồm bảng validation, metric từng seed và ensemble, lệnh đã chạy, hash, commit; checkpoint V4 của từng seed. Từ chối ghi đè.

In [ ]:
DRIVE_RUN_ROOT.mkdir(parents=True, exist_ok=True)
run_dir_drive = DRIVE_RUN_ROOT / RUN_ID
run_zip = DRIVE_RUN_ROOT / f'{RUN_ID}.zip'
if run_dir_drive.exists() or run_zip.exists():
    raise FileExistsError(f'Từ chối ghi đè: {run_dir_drive} hoặc {run_zip}')
payload = run_dir_drive / 'payload'
payload.mkdir(parents=True)
(payload / 'summary_validation_pairs.json').write_text(json.dumps(rows, indent=2), encoding='utf-8')
(payload / 'commands.json').write_text(json.dumps(COMMANDS, indent=2), encoding='utf-8')
for seed in SEEDS:
    target = payload / f'seed{seed}'
    target.mkdir()
    for name in ('v4_metrics.json', 'init_metrics.json', 'test_per_class.csv', 'test_predictions.csv'):
        source = VAL_EVAL[seed]['dir'] / name
        if source.is_file():
            shutil.copy2(source, target / f'val_{name}')
    shutil.copy2(V4_RUNS[seed] / 'history.json', target / 'history.json')
    shutil.copy2(V4_RUNS[seed] / 'best.pt', target / 'v4_best.pt')
for path in ENS_DIR.rglob('*.json'):
    destination = payload / 'ensembles' / path.relative_to(ENS_DIR)
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(path, destination)
if RUN_LOCKED_TEST:
    for seed in SEEDS:
        for name in ('v4_metrics.json', 'init_metrics.json'):
            shutil.copy2(TEST_RESULTS[seed]['dir'] / name, payload / f'seed{seed}' / f'test_{name}')
hashes = {'repo_commit': REPO_COMMIT_RESOLVED, 'train_val_artifact_source': str(TRAIN_VAL_ARTIFACT_ZIP),
          'drive_manifest_sha256': {split: sha256_file(path) for split, path in DRIVE_MANIFESTS.items()},
          'v4_checkpoints': {f'seed{seed}': sha256_file(V4_RUNS[seed] / 'best.pt') for seed in SEEDS},
          'init_checkpoints': {f'seed{seed}': sha256_file(SEED_INITS[seed]) for seed in SEEDS}}
(payload / 'hashes.json').write_text(json.dumps(hashes, indent=2), encoding='utf-8')
(run_dir_drive / 'run_manifest.json').write_text(json.dumps({
    'run_id': RUN_ID, 'created_at_utc': dt.datetime.now(dt.timezone.utc).isoformat(), 'repo_commit': REPO_COMMIT_RESOLVED,
    'seeds': SEEDS, 'locked_test': RUN_LOCKED_TEST,
    'protocol': 'V4 initialised from the matching c_ref seed checkpoint; paired validation comparison; test only if enabled (exploratory)'},
    indent=2), encoding='utf-8')
with zipfile.ZipFile(run_zip, 'x', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(run_dir_drive.rglob('*')):
        if path.is_file():
            archive.write(path, arcname=path.relative_to(run_dir_drive))
print('Run ZIP:', run_zip)
